# Lesson 26 Lab — A Hypothesis-Driven Tuning Loop

**Puzzle:** Which engine knob should change first when throughput is low but p95 is already close to the SLO?

This notebook retains the output of a complete RTX 5090 run.


## Why this matters

Tuning many flags at once produces an irreproducible winner. A disciplined loop starts from a bottleneck hypothesis, changes one factor, records raw evidence, and rejects improvements that violate another gate.


## 0. Predict before running

1. Predict where throughput begins to saturate.
2. Identify the held-constant variables.
3. Apply the gate before reading the candidate name.

For every answer, name the observation that would disprove it.


## 1. Name the concrete objects

The native lab holds one engine and prompt family constant while sweeping batch sizes 1, 2, 4, and 8. It calculates output throughput and a latency proxy, then applies a declared throughput/p95 decision rule.

- One experiment should test one written hypothesis.
- Every candidate retains all acceptance metrics.
- A throughput gain outside the latency gate is not a winner.


## 2. Derive the mechanism

Batching can amortize weight reads and raise occupancy, but closed-batch completion latency grows with more work. Engine limits such as maximum sequences, maximum batched tokens, memory utilization, eager execution, and compilation interact with workload shape. A Pareto frontier is more useful than one scalar score.

### Mechanism at a glance

```mermaid
flowchart LR
  B["baseline evidence"] --> H["one bottleneck hypothesis"]
  H --> C["change one variable"]
  C --> M["measure raw samples"]
  M --> G{"all gates pass?"}
  G -->|"yes"| P["candidate on Pareto frontier"]
  G -->|"no"| R["reject and preserve baseline"]
  P --> H
```

### Walk it step by step

1. **Write the hypothesis.** Name the bottleneck and expected metric movement.
2. **Freeze the comparison.** Change one engine or workload variable.
3. **Retain the distribution.** Keep samples, resource data, and error counts.
4. **Apply all gates.** Select from feasible Pareto rows and preserve rollback.


## 3. Inspect the execution environment

The next cell asserts CUDA, prints the RTX 5090/PyTorch/CUDA/vLLM identity, fixes a seed, and defines only the helpers used by this chapter.


In [1]:
LESSON_NO = 26
LESSON_TITLE = 'A Hypothesis-Driven Tuning Loop'

from pathlib import Path
import gc, hashlib, importlib, inspect, ipaddress, json, math, os, random, re
import shutil, statistics, subprocess, sys, tempfile, time
from urllib.parse import urlparse

# The default FlashInfer sampler requires a local JIT link setup that is not
# guaranteed in wheel-only environments. vLLM's native PyTorch sampler keeps
# these labs reproducible without changing attention or scheduling backends.
os.environ.setdefault("VLLM_USE_FLASHINFER_SAMPLER", "0")

import requests
import torch
import vllm
import yaml
from vllm import LLM, SamplingParams

assert torch.cuda.is_available(), "This lab requires a CUDA-capable GPU."
DEVICE = torch.device("cuda")
SEED = 20260812 + LESSON_NO
random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
MODEL_PATH = os.environ.get("CH3_MODEL", "Qwen/Qwen2.5-1.5B-Instruct")
MODEL = Path(MODEL_PATH)
assert MODEL.exists(), f"Set CH3_MODEL to a local model directory; not found: {MODEL}"

gpu_name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
ENV = {
    "gpu": gpu_name, "compute_capability": f"{major}.{minor}",
    "torch": torch.__version__, "cuda_runtime": str(torch.version.cuda),
    "python": sys.version.split()[0], "vllm": vllm.__version__,
    "model_path": MODEL.name, "seed": SEED,
}
print(json.dumps(ENV, indent=2))

def percentile(values, q):
    ordered = sorted(float(v) for v in values)
    if not ordered: return float("nan")
    pos = (len(ordered) - 1) * q; lo, hi = math.floor(pos), math.ceil(pos)
    return ordered[lo] if lo == hi else ordered[lo] * (hi - pos) + ordered[hi] * (pos - lo)

def model_config():
    return json.loads((MODEL / "config.json").read_text(encoding="utf-8"))

def base_engine_args(**overrides):
    values = {
        "model": str(MODEL), "tokenizer": str(MODEL), "trust_remote_code": False,
        "dtype": "bfloat16", "max_model_len": 2048, "gpu_memory_utilization": 0.45,
        "enforce_eager": True, "seed": SEED, "max_num_seqs": 16,
    }
    values.update(overrides); return values

def output_record(item):
    completion = item.outputs[0]; tokens = list(completion.token_ids)
    return {
        "request_id": str(item.request_id), "prompt_tokens": len(item.prompt_token_ids or []),
        "output_tokens": len(tokens), "token_ids": tokens, "text_preview": completion.text[:120],
        "text_sha256": hashlib.sha256(completion.text.encode()).hexdigest(),
        "finish_reason": str(completion.finish_reason),
        "stop_reason": None if completion.stop_reason is None else str(completion.stop_reason),
        "num_cached_tokens": int(getattr(item, "num_cached_tokens", 0) or 0),
    }

def vllm_cli():
    candidate = Path(sys.executable).parent / "vllm"
    return str(candidate if candidate.exists() else (shutil.which("vllm") or "vllm"))

def cli_help(*args):
    result = subprocess.run([vllm_cli(), *args, "--help"], capture_output=True, text=True, timeout=60)
    return result.returncode, result.stdout + result.stderr

def run_server_probe(port, request_payload=None, scrape_metrics=False):
    log_path = Path(tempfile.gettempdir()) / f"ch03-vllm-{LESSON_NO}-{port}.log"
    command = [vllm_cli(), "serve", str(MODEL), "--host", "127.0.0.1", "--port", str(port),
               "--dtype", "bfloat16", "--max-model-len", "1024", "--gpu-memory-utilization", "0.45",
               "--enforce-eager", "--disable-uvicorn-access-log"]
    started = time.perf_counter()
    with log_path.open("w", encoding="utf-8") as log:
        process = subprocess.Popen(command, stdout=log, stderr=subprocess.STDOUT, text=True)
    ready = False
    try:
        deadline = time.time() + 300
        while time.time() < deadline:
            if process.poll() is not None: break
            try:
                if requests.get(f"http://127.0.0.1:{port}/health", timeout=2).status_code == 200:
                    ready = True; break
            except requests.RequestException: pass
            time.sleep(1)
        startup_s = time.perf_counter() - started
        if not ready:
            raise RuntimeError("vLLM server failed to start:\n" + log_path.read_text(errors="replace")[-6000:])
        models = requests.get(f"http://127.0.0.1:{port}/v1/models", timeout=30)
        data = {"server_ready": True, "startup_s": startup_s,
                "models_status": models.status_code, "model_json": models.json()}
        if request_payload is not None:
            tick = time.perf_counter()
            chat = requests.post(f"http://127.0.0.1:{port}/v1/chat/completions",
                                 json=request_payload, timeout=180)
            data.update(chat_status=chat.status_code, chat_latency_s=time.perf_counter() - tick,
                        chat_json=chat.json())
        if scrape_metrics:
            response = requests.get(f"http://127.0.0.1:{port}/metrics", timeout=30)
            data.update(metrics_status=response.status_code, metrics_text=response.text)
        return data
    finally:
        if process.poll() is None:
            process.terminate()
            try: process.wait(timeout=30)
            except subprocess.TimeoutExpired: process.kill(); process.wait(timeout=10)
        tail = log_path.read_text(errors="replace")[-4000:] if log_path.exists() else ""
        private_home = "/" + "root" + "/"
        globals()["SERVER_LOG_TAIL"] = tail.replace(str(MODEL), "$CH3_MODEL").replace(private_home, "<remote-home>/")


<remote-home>/vllm-ch03/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "torch": "2.13.0+cu130",
  "cuda_runtime": "13.0",
  "python": "3.12.3",
  "vllm": "0.27.1",
  "model_path": "Qwen2.5-1.5B-Instruct",
  "seed": 20260838
}


## 4. Freeze the comparison

| Role | Frozen value |
|---|---|
| Baseline | batch size one |
| Candidate | batch sizes two, four, and eight under one engine |
| Held constant | model, prompts, token limit, sampling, warm-up, repeats, and GPU |
| Measurements | raw elapsed samples, output tokens/s, batch completion p95, memory, feasible set, and selected row |
| Evidence | `native-backend` |

**Experiment:** Run a native batch-size sweep and select only Pareto/gate-feasible rows.


## 5. Inspect the experiment code

The code stores raw samples for every cell and calculates the gate from data. It does not tune engine construction flags in the same experiment.

Do not execute until the code matches the frozen table.


In [2]:
llm=LLM(**base_engine_args(max_model_len=1024,max_num_seqs=16)); params=SamplingParams(temperature=0.0,max_tokens=20,seed=SEED)
llm.generate(["warmup"],SamplingParams(temperature=0.0,max_tokens=2),use_tqdm=False); rows=[]; torch.cuda.reset_peak_memory_stats()
for batch in (1,2,4,8):
    samples=[]; output_count=0
    for repeat in range(3):
        prompts=[f"{repeat}-{i}: name one inference bottleneck." for i in range(batch)]
        tick=time.perf_counter(); outputs=llm.generate(prompts,params,use_tqdm=False); samples.append(time.perf_counter()-tick)
        output_count=sum(len(x.outputs[0].token_ids) for x in outputs)
    median=statistics.median(samples); rows.append({"batch":batch,"samples_s":samples,"median_s":median,
        "p95_s":percentile(samples,.95),"output_tokens_s":output_count/median})
gate=max(row["p95_s"] for row in rows[:2])*2.5; feasible=[row for row in rows if row["p95_s"]<=gate]
selected=max(feasible,key=lambda row:row["output_tokens_s"])
metrics={"candidates":len(rows),"rows":rows,"p95_gate_s":gate,"feasible_candidates":len(feasible),
 "selected":selected,"peak_allocated_mib":torch.cuda.max_memory_allocated()/2**20}
analysis=(f"The one-variable sweep kept {len(feasible)}/{len(rows)} rows below the {gate:.3f} s "
          f"closed-batch p95 gate. Batch {selected['batch']} led feasible throughput at "
          f"{selected['output_tokens_s']:.1f} output tokens/s; online latency is not inferred.")


INFO 08-13 00:24:52 [api_utils.py:273] non-default args: {'tokenizer': '<remote-home>/autodl-tmp/models/Qwen/Qwen2.5-1.5B-Instruct', 'dtype': 'bfloat16', 'seed': 20260838, 'max_model_len': 1024, 'gpu_memory_utilization': 0.45, 'max_num_seqs': 16, 'disable_log_stats': True, 'enforce_eager': True, 'model': '<remote-home>/autodl-tmp/models/Qwen/Qwen2.5-1.5B-Instruct'}


INFO 08-13 00:24:52 [model.py:645] Resolved architecture: Qwen2ForCausalLM


INFO 08-13 00:24:52 [model.py:1883] Using max model len 1024


INFO 08-13 00:24:52 [scheduler.py:242] Chunked prefill is enabled with max_num_batched_tokens=8192.


WARNING 08-13 00:24:52 [vllm.py:1194] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none


...

(EngineCore pid=653775) 2026-08-13 00:25:05,400 - INFO - autotuner.py:2397 - flashinfer.jit: [Autotuner]: Loaded 0 configs from <remote-home>/.cache/vllm/flashinfer_autotune_cache/flashinfer/0.6.16.post3/d10e66a551b175d65f9f24bcac568452ca3ec2ddfb6e3ee96a3fcc8c0723996c/autotune_configs.json
(EngineCore pid=653775) 2026-08-13 00:25:05,400 - INFO - autotuner.py:829 - flashinfer.jit: [Autotuner]: Autotuning process starts ...
(EngineCore pid=653775) 2026-08-13 00:25:05,459 - INFO - autotuner.py:852 - flashinfer.jit: [Autotuner]: Autotuning process ends
(EngineCore pid=653775) 2026-08-13 00:25:05,465 - INFO - autotuner.py:2269 - flashinfer.jit: [Autotuner]: Saved 0 configs to <remote-home>/.cache/vllm/flashinfer_autotune_cache/flashinfer/0.6.16.post3/d10e66a551b175d65f9f24bcac568452ca3ec2ddfb6e3ee96a3fcc8c0723996c/autotune_configs.json (0 new, 0 from previous config)


(EngineCore pid=653775) INFO 08-13 00:25:06 [jit_monitor.py:79] Kernel JIT monitor activated; monitored JIT compilations during inference will use mode=warn.


(EngineCore pid=653775) INFO 08-13 00:25:06 [core.py:355] init engine (profile, create kv cache, warmup model) took 2.75 s


(EngineCore pid=653775) WARNING 08-13 00:25:06 [vllm.py:1194] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
(EngineCore pid=653775) WARNING 08-13 00:25:06 [vllm.py:1247] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
(EngineCore pid=653775) INFO 08-13 00:25:06 [kernel.py:306] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['vllm_c', 'native'], fused_add_rms_norm=['vllm_c', 'native'])
(EngineCore pid=653775) INFO 08-13 00:25:06 [vllm.py:1426] Cudagraph is disabled under eager mode
(EngineCore pid=653775) INFO 08-13 00:25:06 [compilation.py:329] Enabled custom fusions: norm_quant, act_quant


INFO 08-13 00:25:07 [hf.py:540] Detected the chat template content format to be 'string'. You can set `--chat-template-content-format` to override this.


## 6. Read the retained RTX 5090 result

**Recorded environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.13.0+cu130; CUDA runtime 13.0; vLLM 0.27.1.

| Measured field | Checked-in value |
|---|---:|
| Candidates | 4 |
| Feasible candidates | 4 |
| Selected batch | 8 |
| Selected throughput | 925.4/s |
| Selected p95 | 0.180912 |
| Peak allocated | 0.000 MiB |


## 7. Explain the result

The one-variable sweep kept 4/4 rows below the 0.426 s closed-batch p95 gate. Batch 8 led feasible throughput at 925.4 output tokens/s; online latency is not inferred.

This interpretation is bounded to the printed model, GPU, packages, workload, and evidence label.


## 8. Keep the evidence label honest

This run is labeled **`native-backend`**. The named vLLM runtime executed on the recorded GPU/model/workload. The result does not transfer to another version, model, endpoint, or traffic distribution.

The next cell writes and prints the canonical JSON artifact.


In [3]:
artifact = Path("artifacts/rtx5090-result.json")
artifact.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "lesson": 26, "title": 'A Hypothesis-Driven Tuning Loop', "environment": ENV,
    "evidence_label": 'native-backend', "metrics": metrics,
    "analysis": analysis, "conclusion": 'A tuning result is a gated comparison with frozen variables and raw evidence, not a collection of unexplained flags.',
}
artifact.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(payload, indent=2, ensure_ascii=False))


{
  "lesson": 26,
  "title": "A Hypothesis-Driven Tuning Loop",
  "environment": {
    "gpu": "NVIDIA GeForce RTX 5090",
    "compute_capability": "12.0",
    "torch": "2.13.0+cu130",
    "cuda_runtime": "13.0",
    "python": "3.12.3",
    "vllm": "0.27.1",
    "model_path": "Qwen2.5-1.5B-Instruct",
    "seed": 20260838
  },
  "evidence_label": "native-backend",
  "metrics": {
    "candidates": 4,
    "rows": [
      {
        "batch": 1,
        "samples_s": [
          0.15541153494268656,
          0.15411847503855824,
          0.1518243169412017
        ],
        "median_s": 0.15411847503855824,
        "p95_s": 0.15528222895227373,
        "output_tokens_s": 129.77029519008858
      },
      {
        "batch": 2,
        "samples_s": [
          0.1692607388831675,
          0.17057588510215282,
          0.16858342802152038
        ],
        "median_s": 0.1692607388831675,
        "p95_s": 0.1704443704802543,
        "output_tokens_s": 236.3217853350508
      },
      {
      

## 9. Make the bounded decision

> A tuning result is a gated comparison with frozen variables and raw evidence, not a collection of unexplained flags.

**Acceptance/rollback:** Promote the smallest-complexity candidate that materially improves the target metric while every quality, latency, memory, and error gate passes.

**Failure analysis:** Closed batches do not measure TTFT/ITL or arrival queues. Batch-size effects can differ after compilation, prefix caching, quantization, or longer outputs.


## 10. Extend the evidence

Choose the next single-variable hypothesis from a native profile, then repeat with open-loop service traffic and confidence intervals.

The full boundary and references are in [`README.md`](README.md).
